# Experiment: Cross-Experiment Error Analysis

This notebook reads cached analysis outputs produced by `scripts/error_analysis.py` and focuses on failure modes across `wordorder`, `size`, `orthography`, and `agreement`.

Questions:
- How much of the bag-of-words vs exact-match gap is really word-order error?
- Which other failure modes recur across experiments and models?
- Which failure modes are experiment-specific rather than generic degradation?


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
OUT_DIR = PROJECT_ROOT / "notebooks" / "cache" / "error-analysis"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(NOTEBOOKS_DIR))

import aesthetics as aes  # noqa: E402

rows_df = pd.read_csv(OUT_DIR / "rows.csv")
metric_df = pd.read_csv(OUT_DIR / "metric_summary.csv")
failure_df = pd.read_csv(OUT_DIR / "failure_mode_summary.csv")
orth_df = pd.read_csv(OUT_DIR / "orthography_summary.csv")
example_rows = json.loads((OUT_DIR / "example_rows.json").read_text())

large_datasets = {"wordorder_large_exp", "orthography_large_exp"}
rows_df = rows_df.loc[rows_df["dataset"].isin(large_datasets)].copy()
metric_df = metric_df.loc[metric_df["dataset"].isin(large_datasets)].copy()
failure_df = failure_df.loc[failure_df["dataset"].isin(large_datasets)].copy()
orth_df = orth_df.loc[orth_df["dataset"].isin(large_datasets)].copy()
model_order = aes.ordered_models(rows_df["fuzzy_model"].dropna().unique())

rows_df.shape, metric_df.shape, failure_df.shape

## Coverage

The cached table spans all currently available model outputs in the repository, including the agreement compact batches.


In [ ]:
coverage_df = (
    rows_df.groupby(["exp", "fuzzy_model"]).size().rename("rows").reset_index()
)
coverage_df["fuzzy_model"] = pd.Categorical(
    coverage_df["fuzzy_model"], categories=model_order, ordered=True
)
metric_df["fuzzy_model"] = pd.Categorical(
    metric_df["fuzzy_model"], categories=model_order, ordered=True
)
display(coverage_df.sort_values(["exp", "fuzzy_model"]))
display(metric_df.sort_values(["exp", "fuzzy_model"]))

## Metric Gaps

A useful sanity check: the bag-of-words improvement is often much smaller than the total error mass. That means models are failing in ways other than pure permutation.


In [ ]:
plot_df = metric_df.copy()
plot_df["exact_match"] *= 100
plot_df["bow_match"] *= 100
plot_df["bow_minus_exact"] *= 100

display(
    plot_df[
        ["exp", "fuzzy_model", "exact_match", "bow_match", "bow_minus_exact"]
    ].sort_values(["exp", "fuzzy_model"])
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(
    data=plot_df,
    x="exp",
    y="bow_minus_exact",
    hue="fuzzy_model",
    hue_order=model_order,
    palette=aes.PALETTE_MODELS,
    ax=ax,
)
ax.set_ylabel("Bag-of-words minus exact match (percentage points)")
ax.set_xlabel("")
ax.set_title("How much error is explained by word order alone?")
plt.xticks(rotation=0)
plt.tight_layout()

## Failure Modes

The main qualitative surprise from the first pass is that several strong failure modes are not just tokenization or permutation:
- `same_length_substitution`: the model outputs an in-vocabulary sentence of the right length, but swaps one or more lexical choices.
- `source_lexicon_intrusion` / `copied_source`: source-language material leaks into the answer.
- `partial_span`: the output is a clean prefix or suffix of the target, suggesting truncation or early stopping.
- `repetition_loop`: local collapse or fallback loops.
- `hallucinated_vocab`: the model invents target-side words not licensed by the grammar.


In [ ]:
display(
    failure_df.sort_values(
        ["exp", "fuzzy_model", "count"], ascending=[True, True, False]
    )
)

top_failure_df = (
    failure_df.sort_values(
        ["exp", "fuzzy_model", "count"], ascending=[True, True, False]
    )
    .groupby(["exp", "fuzzy_model"], as_index=False)
    .head(3)
)
display(top_failure_df)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=failure_df,
    x="failure_mode",
    y="pct_within_model_exp",
    hue="exp",
    ax=ax,
)
ax.set_ylabel("Percent of wrong answers")
ax.set_xlabel("")
ax.set_title("Failure mode prevalence across experiments")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

## Orthography-Specific Effects

Orthography remains useful as a special case because it separates lexical mapping errors from script-handling failures. The cached summary tracks wrong-script and diacritic-drop rates directly.


In [ ]:
display(orth_df.sort_values(["fuzzy_model", "target_orthography"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
sns.barplot(
    data=orth_df,
    x="target_orthography",
    y="wrong_script",
    hue="fuzzy_model",
    hue_order=model_order,
    palette=aes.PALETTE_MODELS,
    ax=axes[0],
)
axes[0].set_title("Wrong-script rate")
axes[0].set_ylabel("Rate")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=orth_df,
    x="target_orthography",
    y="diacritic_drop",
    hue="fuzzy_model",
    hue_order=model_order,
    palette=aes.PALETTE_MODELS,
    ax=axes[1],
)
axes[1].set_title("Diacritic-drop rate")
axes[1].set_ylabel("Rate")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

for ax in axes:
    ax.legend(frameon=False)

plt.tight_layout()

## Representative Examples

The cached examples make it easy to inspect each failure type without scanning the full row table.


In [ ]:
def show_examples(mode: str, n: int = 3) -> pd.DataFrame:
    return pd.DataFrame(example_rows[mode]).head(n)


display(show_examples("same_length_substitution"))
display(show_examples("source_lexicon_intrusion"))
display(show_examples("partial_span"))
display(show_examples("repetition_loop"))
display(show_examples("wrong_script"))
display(show_examples("diacritic_drop"))

## Notes

- `agreement` is loaded from the compact batch outputs plus a lightweight prefix parse of the sample manifests so the notebook does not materialize the giant `possible_right*` fields.
- The current failure labels are heuristic, not mutually exhaustive causal diagnoses.
- `hallucinated_vocab` means the prediction contains tokens outside the licensed target-side vocabulary for that grammar.
- The strongest new pattern in the current pass is agreement/word-order/size substitution behavior: many wrong answers are structurally plausible and nearly length-matched, but lexically wrong.
- Re-run `uv run python notebooks/error_analysis.py` to refresh the cached artifacts when new batch outputs land.
